In [1]:
import re
import string

import pandas as pd
import numpy as np

import spacy
from spacy.lang.en.stop_words import STOP_WORDS

In [2]:
df = pd.read_json(
    r"D:/Old laptop data/Udemy/Role Play - NLP Engineer/dataset/News_Category_Dataset_v3.json",
    lines=True
)

print("Dataset shape:", df.shape)

Dataset shape: (209527, 6)


In [3]:
df_clean = df.copy()

In [4]:
print(
    "Duplicate rows before cleaning:",
    df_clean.duplicated().sum()
)

Duplicate rows before cleaning: 13


In [5]:
df_clean = df_clean.drop_duplicates().copy()

print(
    "Duplicate rows after cleaning:",
    df_clean.duplicated().sum()
)

Duplicate rows after cleaning: 0


In [6]:
empty_headlines = (
    df_clean["headline"]
    .fillna("")
    .str.strip()
    .eq("")
)

print("Empty headlines:", empty_headlines.sum())

Empty headlines: 6


In [7]:
df_clean = df_clean.loc[~empty_headlines].copy()

In [8]:
print(
    "Empty headlines after cleaning:",
    df_clean["headline"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

Empty headlines after cleaning: 0


#### Now we will deal with the empty descriptions. As we saw earlier even though the descriptions were empty, the headlines were still present for those entries. We will use those headlines as descriptions as they can provide context.

In [9]:
df_clean["short_description"] = (
    df_clean["short_description"]
    .fillna("")
    .str.strip()
)

In [10]:
df_clean["text"] = (
    df_clean["headline"].str.strip()
    + " "
    + df_clean["short_description"]
).str.strip()

In [13]:
df_clean.tail(5)

,link,headline,category,short_description,authors,date,text
209522,https://www.huffingtonpost.com/entry/rim-ceo-t...,RIM CEO Thorsten Heins' 'Significant' Plans Fo...,TECH,Verizon Wireless and AT&T are already promotin...,"Reuters, Reuters",2012-01-28,RIM CEO Thorsten Heins' 'Significant' Plans Fo...
209523,https://www.huffingtonpost.com/entry/maria-sha...,Maria Sharapova Stunned By Victoria Azarenka I...,SPORTS,"Afterward, Azarenka, more effusive with the pr...",,2012-01-28,Maria Sharapova Stunned By Victoria Azarenka I...
209524,https://www.huffingtonpost.com/entry/super-bow...,"Giants Over Patriots, Jets Over Colts Among M...",SPORTS,"Leading up to Super Bowl XLVI, the most talked...",,2012-01-28,"Giants Over Patriots, Jets Over Colts Among M..."
209525,https://www.huffingtonpost.com/entry/aldon-smi...,Aldon Smith Arrested: 49ers Linebacker Busted ...,SPORTS,CORRECTION: An earlier version of this story i...,,2012-01-28,Aldon Smith Arrested: 49ers Linebacker Busted ...
209526,https://www.huffingtonpost.com/entry/dwight-ho...,Dwight Howard Rips Teammates After Magic Loss ...,SPORTS,The five-time all-star center tore into his te...,,2012-01-28,Dwight Howard Rips Teammates After Magic Loss ...


In [14]:
## Text cleaning function
def clean_text(text):
    """
    Clean news text for NLP classification.
    """

    # Convert to string and lowercase
    text = str(text).lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
## Trying the text cleaning function with an example

sample_text = "Trump's New Policy: What Does It Mean? https://example.com"

cleaned_sample_text = clean_text(sample_text)
print("Original text:", sample_text)
print("Cleaned text:", cleaned_sample_text)

Original text: Trump's New Policy: What Does It Mean? https://example.com
Cleaned text: trumps new policy what does it mean


In [17]:
df_clean["clean_text"] = df_clean["text"].apply(clean_text)

In [18]:
df_clean[["text", "clean_text"]].head(10)

,text,clean_text
0,Over 4 Million Americans Roll Up Sleeves For O...,over 4 million americans roll up sleeves for o...
1,"American Airlines Flyer Charged, Banned For Li...",american airlines flyer charged banned for lif...
2,23 Of The Funniest Tweets About Cats And Dogs ...,23 of the funniest tweets about cats and dogs ...
3,The Funniest Tweets From Parents This Week (Se...,the funniest tweets from parents this week sep...
4,Woman Who Called Cops On Black Bird-Watcher Lo...,woman who called cops on black birdwatcher los...
5,Cleaner Was Dead In Belk Bathroom For 4 Days B...,cleaner was dead in belk bathroom for 4 days b...
6,Reporter Gets Adorable Surprise From Her Boyfr...,reporter gets adorable surprise from her boyfr...
7,Puerto Ricans Desperate For Water After Hurric...,puerto ricans desperate for water after hurric...
8,How A New Documentary Captures The Complexity ...,how a new documentary captures the complexity ...
9,Biden At UN To Call Russian War An Affront To ...,biden at un to call russian war an affront to ...


In [19]:
empty_clean_text = (
    df_clean["clean_text"]
    .str.strip()
    .eq("")
)

print("Empty cleaned texts:", empty_clean_text.sum())

Empty cleaned texts: 0


In [20]:
print(df_clean.shape)

(209508, 8)


In [21]:
df_clean["category"].value_counts().head()

category
POLITICS          35600
WELLNESS          17942
ENTERTAINMENT     17362
TRAVEL             9900
STYLE & BEAUTY     9811
Name: count, dtype: int64

In [23]:
df_clean.to_csv(
    "../dataset/processed/news_cleaned.csv",
    index=False
)